In [1]:
!nvidia-smi

Sun Aug 30 08:12:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.173.02             Driver Version: 580.173.02     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      On  |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install datasets transformers torch accelerate kernels tqdm

Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset

ds = load_dataset("EthanKim8683/reg_grpo_128", split="train")
ds

Dataset({
    features: ['prompt', 'outputs', 'advantages'],
    num_rows: 128
})

In [5]:
def expand_groups(batch):
	new_batch = {
		"prompt": [],
		"output": [],
		"advantage": [],
	}
	for prompt, outputs, advantages in zip(
		batch["prompt"],
		batch["outputs"],
		batch["advantages"],
	):
		for output, advantage in zip(outputs, advantages):
			new_batch["prompt"].append(prompt)
			new_batch["output"].append(output)
			new_batch["advantage"].append(advantage)
	return new_batch

expanded_ds = ds.map(
	expand_groups,
	batched=True,
	remove_columns=ds.column_names
)
expanded_ds

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'output', 'advantage'],
    num_rows: 8192
})

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "openai/gpt-oss-20b"

model = AutoModel.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/410 [00:00<?, ?it/s]

[transformers] GptOssModel LOAD REPORT from: openai/gpt-oss-20b
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch

class AdvHead(torch.nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.linear = torch.nn.Linear(hidden_size, 1, bias=False)

    def forward(self, x):
        x = self.linear(x)
        return torch.asinh(x)

adv_head = AdvHead(model.config.hidden_size)

In [ ]:
from tqdm import tqdm
from accelerate import Accelerator
import gc

NUM_EPOCHS = 3

accelerator = Accelerator(mixed_precision="bf16")

loss_fn = torch.nn.SmoothL1Loss(reduction="none")
optim = torch.optim.AdamW(adv_head.parameters(), lr=1e-4)

(
	model,
	adv_head,
	optim,
) = accelerator.prepare(
	model,
	adv_head,
	optim,
)

model.eval()

for epoch in range(NUM_EPOCHS):
	progress_bar = tqdm(
		expanded_ds.shuffle(),
		desc=f"Epoch {epoch+1}/{NUM_EPOCHS}",
		disable=not accelerator.is_main_process,
	)
	for example in progress_bar:
		optim.zero_grad()

		inputs = tokenizer.apply_chat_template(
			[{"role": "user", "content": example["prompt"]}],
			add_generation_prompt=True,
		)

		with torch.no_grad():
			outputs = model(**inputs)
			last_hidden_state = outputs.last_hidden_state
		
		pred_advs = adv_head(last_hidden_state).squeeze(-1)
		target_advs = example["advantage"].unsqueeze(-1).expand_as(pred_advs)
		loss = loss_fn(pred_advs, target_advs)
		
		accelerator.backward(loss)
		if accelerator.sync_gradients:
			accelerator.clip_grad_norm_(adv_head.parameters(), max_norm=1.0)
		optim.step()

		progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

		del (
			outputs,
			last_hidden_state,
			pred_advs,
			target_advs,
			loss,
		)
		gc.collect()
		torch.cuda.empty_cache()

torch.Size([1, 3951])


torch.Size([1, 4594])


[W829 23:48:09.384445297 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 2703228928 bytes (free: 2110914560, total: 23659151360).
[W829 23:48:09.545815261 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 2703228928 bytes (free: 2182217728, total: 23659151360).
Epoch 1/3:   0%|          | 1/8192 [00:12<27:57:26, 12.29s/it, loss=1.0631]


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.52 GiB. GPU 0 has a total capacity of 22.03 GiB of which 2.03 GiB is free. Including non-PyTorch memory, this process has 19.99 GiB memory in use. Of the allocated memory 17.09 GiB is allocated by PyTorch, and 2.67 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)